In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

In [ ]:
import json
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

from maintenance_llm.llm_client.gpt_client import gpt_client
from maintenance_llm.prompts.prompt_builder import build_prompt
from maintenance_llm.rag_module.bge import bge_retriever
from maintenance_llm.rag_module.context_compiler import compile_context
from maintenance_llm.rag_module.minilm import minilm_retriever
from maintenance_llm.rag_module.tfidf import tfidf_retriever
from maintenance_llm.utils.output_parser import batch_parser
from maintenance_llm.utils.paths import BATCH_DIR, LABEL_DATABASE_FILE

load_dotenv()

Initialise Notebook-based Functions

In [ ]:
"""Function to initialise the selected retriever based on the provided name and context database"""

def get_retriever(retriever_name: str, context_db: list[str]):
    retrievers = {
        "tfidf": tfidf_retriever,
        "minilm": minilm_retriever,
        "bge": bge_retriever,
        
    }

    if retriever_name not in retrievers:
        raise ValueError(
            f"Unsupported retriever: {retriever_name}"
        )

    return retrievers[retriever_name](context_db)

In [ ]:
"""Function to prepare and compile prompts(default/RAG) with corresponding custom_id into a pd.DataFrame for batch inference"""

def get_prompt_batch(dataset: pd.DataFrame, retriever_model: str, RAG: bool = False) -> pd.DataFrame:
    # Prepare input texts and valid labels for prediction
    input_texts = dataset['ServiceOrderText'].tolist()
    valid_labels = dataset['Failure Category'].unique().tolist()
    
    if RAG: 
        # Load context database from json 
        with open(LABEL_DATABASE_FILE, "r", encoding= "utf-8") as f: 
            context_data = json.load(f)
                        
        context_db = compile_context(context_data) # compile context data into formatted list
        retriever = get_retriever(retriever_model, context_db)  # initialise selected retriever
        retrieved_df = retriever.retrieve_context(input_texts)  # returns df of input_texts and corresponding top k context data 
        
        inference_df = retrieved_df

    else: 
        inference_df = pd.DataFrame({"input_text": input_texts})

    inference_df.insert(0, "WO_ID", dataset["Work Order ID"].tolist())

    # compile prompt for batch_input file
    prompt_df = []
    for _, row in inference_df.iterrows():
        WO_id = row["WO_ID"]
        text = row["input_text"]
        if RAG: 
            prompt = build_prompt(rag= RAG, text=text, retrieved_context= row["retrieved_context"], labels=valid_labels)
        else: 
            prompt = build_prompt(rag= RAG, text=text, labels=valid_labels)
        
        prompt_df.append(
            {
                "WO_ID": WO_id,
                "prompt": prompt,
            }
        )
        
    return pd.DataFrame(prompt_df)

In [ ]:
"""Function to split inference dataframe into n batches based on the maximum concurrent token limit for GPT Batch API"""

def sort_batch_size(prompt_df:pd.DataFrame, max_batch_tokens:int) -> list[dict]:
    batches = []
    current_batch = []
    current_tokens = 0

    for _, row in prompt_df.iterrows():
        wo_id = row["WO_ID"]
        prompt = row["prompt"]
        token_count = row["input_tokens"]

        if current_tokens + token_count > max_batch_tokens:
            batches.append(current_batch)
            current_batch = []
            current_tokens = 0

        current_batch.append({
            "WO_ID": wo_id,
            "prompt": prompt,
            "input_tokens": token_count
        })

        current_tokens += token_count

    if current_batch:
        batches.append(current_batch)
        
    return batches


In [ ]:
"""Main Function to run batch inference on the provided dataset using the specified GPT model and retriever, returns the initialised GPT client and the list of created batch files"""

def run_batch_inference(model_name:str, dataset: pd.DataFrame, retriever_model: str, RAG: bool = False, json_format=False, max_batch_tokens: int = 2e6, split_batch: bool = True) -> tuple:
    # Initiate run
    print(f"Run initiated for gpt model: {model_name}")
    print(f"Analysis: batch inferencing | RAG: {RAG} | JSON format: {json_format}")

    print(f"initiating batching for {len(dataset['ServiceOrderText'].tolist())} sampled texts... \n")
    
    prompt_df = get_prompt_batch(dataset, retriever_model, RAG)

    valid_labels = dataset['Failure Category'].unique().tolist()
    GPT_client = gpt_client(model = model_name, valid_labels = valid_labels) # initialise GPT_client

    if split_batch:
        # calculate input token size for batch organisation
        for idx, row in prompt_df.iterrows():
            prompt = row["prompt"]
            input_tokens = GPT_client.count_input_tokens(prompt, json_format=json_format)
            
            prompt_df.loc[idx, "input_tokens"] = input_tokens

        # organise batch based on model token limitation
        prompt_batches = sort_batch_size(prompt_df, max_batch_tokens= max_batch_tokens)
    
    else:
        prompt_batches = [prompt_df.to_dict(orient="records")]
    
    # perform client batch creation based on sorted batch size
    created_batch_files = []
    
    for batch_idx, prompt_batch in enumerate(prompt_batches, start = 1):
        # create batch_file with the compiled prompt
        batch_input_file_path = GPT_client.create_batch_file(prompt_batch, batch_number= batch_idx, json_format=json_format)
        created_batch_files.append(batch_input_file_path)
    
    print("Total batch(es) required to analyse dataset: ", len(created_batch_files))

    return GPT_client, created_batch_files


Load Dataset

In [ ]:
# load dataset
dataset = pd.read_csv(f"{BASE_DIR}/results_public/data_example/dataset_example.csv")
dataset.shape

Inference configurations

In [ ]:
# set run parameter configurations
rag_toggle = True
json_toggle = True
retriever = "tfidf" 
model ="gpt-5.6-luna" # amend GPT model based on user preference
max_batch_tokens = 1_500_000 # amend GPT Batch API token limit based on user limit, better to offset the maximum limit to avoid batch failure (max: 2,000,000 tokens)

Initialise GPT_client and Create Batch Files 

In [ ]:
"""
Note: 
- The splitting of the dataset in batches involves calculation using API-side processing of input token size for each prompt.
- This step may take a while depending on the number of batch files created and the size of the dataset.
- If the user is using a free-tier GPT API account, the maximum token limit is 2,000,000 tokens per batch.
- If the user is confident that the dataset is small enough to fit within the maximum token limit, set split_batch to False to skip batch splitting
"""

# Initialise GPT client and create batch_files based on model token limit
GPT_client, created_batch_list = run_batch_inference(model, dataset, retriever, rag_toggle, json_toggle, max_batch_tokens, split_batch = True)

Batch Submission and Result Retrieval Example (created_batch_list, n = 2)

In [ ]:
"""
Notes: 
1. Batch files should be submitted one by one as they were compiled based on concurrent token limit for GPT Batch API
2. The next batch file should only be submitted once the previous batch has been completed, otherwise the batch will fail
"""

batch_1 = GPT_client.submit_batch(created_batch_list[0])

In [ ]:
# Run to check status and save output file when batch completes 
batch_result_1 = GPT_client.retrieve_batch_results(batch_number=1, batch_id=batch_1.id)

In [ ]:
# Submit the next batch once the previous batch has completed
batch_2 = GPT_client.submit_batch(created_batch_list[1])

In [ ]:
# Run to check status and save output file when batch 2 completes 
batch_result_2 = GPT_client.retrieve_batch_results(batch_number=2, batch_id=batch_2.id)

Extract and combine batch output jsonl files into one csv file

In [ ]:
# import batch output files based on number of files
batch_output_file_1 = (
    Path(BASE_DIR)
    /"reports"
    /"gpt_batch"
    /"batch_output"
    /"batch_output_1_batch_[batch_id].jsonl"
)

batch_output_file_2 = (
    Path(BASE_DIR)
    /"reports"
    /"gpt_batch"
    /"batch_output"
    /"batch_output_2_batch_[batch_id].jsonl"
)

output_list = [batch_output_file_1, batch_output_file_2]

In [ ]:
# extract jsonl to records 
records = []

for files in output_list:
    with open(files, "r") as f:
        for line in f:
            record = json.loads(line)
            records.append(record)
            
# parse batch output records into a single dataframe and save as csv file (user should amend the file name to reflect the batch number and batch_id)
batch_output = batch_parser(records, dataset, json_format=json_toggle)
batch_output.to_csv(BATCH_DIR / "batch_output" / f"base_tfidf_gpt5.6luna_parsed_output_file_batch_[batch_id]_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv") 